# KMeans++ 初始化与收敛

**面试回答：**KMeans 交替分配与更新中心，inertia 单调不增却可能落入局部最优；KMeans++ 用距离平方抽样初始中心。实验是商家客单价与复购率分群，仅验证机制。

## 真实案例

8 家商户的客单价和 30 天复购率用于运营画像；标签不存在，簇只是一种几何划分。

In [1]:
import numpy as np  # 导入 NumPy 手写 KMeans。
name = np.array(['面馆','快餐','奶茶','咖啡','火锅','牛排','甜品','寿司'])  # 构造具名商户。
x = np.array([[25,.18],[31,.20],[19,.25],[42,.38],[116,.12],[168,.10],[53,.31],[91,.27]],dtype=float)  # 记录客单价和复购率。
print('商户 | 客单价 | 复购率')  # 输出原始样本表头。
for n,row in zip(name,x):  # 逐条展示商户画像。
    print(f'{n:2s} | {row[0]:6.0f} | {row[1]:.2f}')  # 输出一条业务样本。

商户 | 客单价 | 复购率
面馆 |     25 | 0.18
快餐 |     31 | 0.20
奶茶 |     19 | 0.25
咖啡 |     42 | 0.38
火锅 |    116 | 0.12
牛排 |    168 | 0.10
甜品 |     53 | 0.31
寿司 |     91 | 0.27


## Baseline / 基线

错误基线把两个中心都放在低价商户，比较其初始 inertia。

In [2]:
def assign(data,centers):  # 定义最近中心分配函数。
    distance=((data[:,None,:]-centers[None,:,:])**2).sum(axis=2)  # 计算所有样本到中心的平方距离。
    return distance.argmin(axis=1),distance  # 返回簇编号与距离矩阵。
bad_centers=x[[0,1]].copy()  # 故意选择相近的坏初始化。
bad_group,bad_distance=assign(x,bad_centers)  # 对坏中心进行首次分配。
bad_inertia=float(bad_distance[np.arange(len(x)),bad_group].sum())  # 计算坏初始化的簇内平方和。
print(f'坏初始化 inertia={bad_inertia:.2f}，中心={bad_centers.tolist()}')  # 输出基线结果。

坏初始化 inertia=30235.07，中心=[[25.0, 0.18], [31.0, 0.2]]


In [3]:
def kmeans_plus_plus(data,k):  # 定义确定性的 KMeans++ 初始化。
    centers=[data[0].copy()]  # 将第一家商户作为第一个中心。
    while len(centers)<k:  # 循环选择剩余中心。
        current=np.vstack(centers)  # 堆叠已选择中心。
        nearest=((data[:,None,:]-current[None,:,:])**2).sum(axis=2).min(axis=1)  # 计算每个样本到最近中心的距离平方。
        centers.append(data[nearest.argmax()].copy())  # 选择距离最大的样本以体现距离平方偏好。
    return np.vstack(centers)  # 返回初始中心矩阵。
def run_kmeans(data,centers):  # 定义分配—均值更新循环。
    history=[]  # 保存每轮 inertia。
    for step in range(12):  # 限制教学迭代轮数。
        group,distance=assign(data,centers)  # 分配每家商户到最近中心。
        inertia=float(distance[np.arange(len(data)),group].sum())  # 计算当前簇内平方和。
        history.append(inertia)  # 记录收敛轨迹。
        centers=np.vstack([data[group==c].mean(axis=0) for c in range(len(centers))])  # 用簇内均值更新每个中心。
    return group,centers,history  # 返回最终分配、中心和轨迹。
start=kmeans_plus_plus(x,2)  # 生成两个分散的 KMeans++ 中心。
group,centers,history=run_kmeans(x,start)  # 执行手写 KMeans。
print('KMeans++ 初始中心:',start.tolist())  # 输出初始化中间量。
print('inertia 轨迹:',np.round(history,2))  # 输出单调收敛过程。
print('最终中心:',np.round(centers,2))  # 输出商户簇画像。

KMeans++ 初始中心: [[25.0, 0.18], [168.0, 0.1]]
inertia 轨迹: [8205.07 4799.53 4799.53 4799.53 4799.53 4799.53 4799.53 4799.53 4799.53
 4799.53 4799.53 4799.53]
最终中心: [[4.35e+01 2.60e-01]
 [1.42e+02 1.10e-01]]


## 结果解读

KMeans++ 从远离首中心的商户选第二中心，通常比相近中心更快进入合理划分；inertia 下降只说明优化目标下降，不证明簇有业务价值。

In [4]:
print('商户 | 簇 | 客单价 | 复购率')  # 输出最终分群表头。
for n,row,c in zip(name,x,group):  # 逐条解释分群。
    print(f'{n:2s} | {c} | {row[0]:6.0f} | {row[1]:.2f}')  # 输出商户分配结果。
print('生产差距：应做特征缩放、异常处理、多随机种子稳定性、簇命名与下游实验。')  # 说明生产边界。

商户 | 簇 | 客单价 | 复购率
面馆 | 0 |     25 | 0.18
快餐 | 0 |     31 | 0.20
奶茶 | 0 |     19 | 0.25
咖啡 | 0 |     42 | 0.38
火锅 | 1 |    116 | 0.12
牛排 | 1 |    168 | 0.10
甜品 | 0 |     53 | 0.31
寿司 | 0 |     91 | 0.27
生产差距：应做特征缩放、异常处理、多随机种子稳定性、簇命名与下游实验。


## 失败案例与修复

若直接把金额和比例放入距离，金额支配分群；修复是用训练集统计量标准化。

In [5]:
z=(x-x.mean(axis=0))/x.std(axis=0)  # 标准化两个量纲不同的特征。
z_start=kmeans_plus_plus(z,2)  # 在标准化空间重新初始化中心。
z_group,z_centers,z_history=run_kmeans(z,z_start)  # 在标准化空间重新收敛。
print('失败：原始空间最终 inertia=',round(history[-1],2))  # 输出量纲混杂的目标量。
print('修复：标准化空间簇=',z_group.tolist())  # 输出尺度一致后的分配。
print('注意：两空间的 inertia 单位不同，不能直接比较大小。')  # 解释正确比较方式。

失败：原始空间最终 inertia= 4799.53
修复：标准化空间簇= [0, 0, 0, 0, 1, 1, 0, 0]
注意：两空间的 inertia 单位不同，不能直接比较大小。


In [6]:
assert len(name)>=5  # 保护真实案例样本数量。
assert history[-1]<=history[0]  # 保护 KMeans 目标单调不增。
assert bad_inertia>history[-1]  # 保护坏初始化的首轮目标更差。
assert len(np.unique(group))==2  # 保护两个簇均非空。